# Módulo M5 --- Ranking híbrido sparse+dense

**Disciplina:** Inteligência Artificial --- FACOM/UFMS --- 2026/1

Combina os *runs* de BM25 (`runs/bm25.trec`) e TF-IDF/KNN (`runs/knn.trec`) via
**Reciprocal Rank Fusion (RRF)**: para cada documento, soma-se $1/(\kappa + rank)$ em
cada sistema em que ele aparece (rank ausente = não contribui). RRF foi escolhido em vez
de soma ponderada de scores porque BM25 e a similaridade do cosseno vivem em escalas
completamente diferentes (não-normalizado vs. $[0,1]$) --- combinar por *posição no
ranking* evita o problema de normalização entre escalas distintas.

In [1]:
import subprocess
from collections import defaultdict
from pathlib import Path

import pandas as pd

## 1. Carregamento dos runs existentes

In [2]:
def load_run(path):
    """Retorna dict: qid -> lista de (doc_id, rank) ordenada por rank."""
    runs = defaultdict(list)
    with open(path) as f:
        for line in f:
            qid, _, doc_id, rank, score, system = line.split()
            runs[qid].append((doc_id, int(rank)))
    for qid in runs:
        runs[qid].sort(key=lambda x: x[1])
    return runs

bm25_runs = load_run("runs/bm25.trec")
knn_runs = load_run("runs/knn.trec")
print("Queries em bm25:", len(bm25_runs), "| Queries em knn:", len(knn_runs))

Queries em bm25: 14 | Queries em knn: 14


## 2. Reciprocal Rank Fusion

$$\text{RRF}(d) = \sum_{\text{sistema} \in \{bm25, knn\}} \frac{1}{\kappa + \text{rank}_{\text{sistema}}(d)}$$

$\kappa$ (tipicamente 60, valor original do paper de Cormack et al.) suaviza a
contribuição de posições muito baixas no ranking. Documentos que aparecem nos dois
sistemas recebem reforço; documentos que aparecem em só um ainda contribuem, mas menos.

In [3]:
KAPPA = 60

def reciprocal_rank_fusion(runs_a, runs_b, kappa=KAPPA, top_k=100):
    qids = set(runs_a) | set(runs_b)
    fused = {}
    for qid in qids:
        scores = defaultdict(float)
        for doc_id, rank in runs_a.get(qid, []):
            scores[doc_id] += 1.0 / (kappa + rank)
        for doc_id, rank in runs_b.get(qid, []):
            scores[doc_id] += 1.0 / (kappa + rank)
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        fused[qid] = ranked
    return fused

hybrid_runs = reciprocal_rank_fusion(bm25_runs, knn_runs)
print("Exemplo (q1), top-5 híbrido:")
for doc_id, score in hybrid_runs["q1"][:5]:
    print(f"  {doc_id}  RRF={score:.5f}")

Exemplo (q1), top-5 híbrido:
  2512.04280  RRF=0.03227
  2502.00031  RRF=0.03089
  2007.03092  RRF=0.03083
  1411.0763  RRF=0.03008
  2209.09090  RRF=0.02991


## 3. Salvando `runs/hybrid.trec`

In [4]:
run_path = Path("runs/hybrid.trec")
with open(run_path, "w", encoding="utf-8") as f:
    for qid, ranked in hybrid_runs.items():
        for rank, (doc_id, score) in enumerate(ranked, 1):
            f.write(f"{qid} Q0 {doc_id} {rank} {score:.6f} hybrid\n")

print("Run híbrida salva em:", run_path)

Run híbrida salva em: runs/hybrid.trec


## 4. Avaliação comparativa: BM25 vs. KNN/TF-IDF vs. Híbrido (RRF)

In [5]:
result = subprocess.run(
    [
        "python3", "../eval/evaluate.py",
        "--qrels", "../eval/qrels.tsv",
        "--runs", "runs/bm25.trec", "runs/knn.trec", "runs/hybrid.trec",
        "--k", "10",
    ],
    capture_output=True, text=True, cwd=".",
)
print(result.stdout)
print(result.stderr)


=== bm25.trec ===
qid   P@10   R@10     AP  nDCG@10
 q1 1.0000 0.4762 0.8877   1.0000
q10 0.6000 0.6667 0.7432   0.7714
q11 0.5000 0.7143 0.8228   0.8105
q12 0.3000 0.5000 0.3349   0.3942
q13 0.3000 1.0000 0.8667   0.9469
q14 0.6000 0.6667 0.6720   0.7213
 q2 0.9000 0.5000 0.7980   0.9364
 q3 0.8000 0.7273 0.7502   0.8580
 q4 1.0000 0.5000 0.9205   1.0000
 q5 0.8000 0.5000 0.6711   0.8482
 q6 0.5000 0.5000 0.6177   0.6047
 q7 0.8000 0.4706 0.7638   0.8522
 q8 0.4000 0.4444 0.5426   0.5683
 q9 0.7000 0.4118 0.6799   0.8007

Medias:
P@10      0.6571
R@10      0.5770
AP        0.7194
nDCG@10   0.7938

=== knn.trec ===
qid   P@10   R@10     AP  nDCG@10
 q1 1.0000 0.4762 0.9010   1.0000
q10 0.7000 0.7778 0.8216   0.8379
q11 0.2000 0.2857 0.2326   0.2651
q12 0.3000 0.5000 0.2709   0.4089
q13 0.3000 1.0000 0.6250   0.8194
q14 0.4000 0.4444 0.4728   0.5629
 q2 0.9000 0.5000 0.8102   0.9266
 q3 0.9000 0.8182 0.9136   0.9149
 q4 1.0000 0.5000 0.8854   1.0000
 q5 0.9000 0.5625 0.9324   0.9337
 q

## 5. Demonstração mínima de uso

Esta lógica também está disponível como script autônomo em `demo.py` (raiz do repositório), que recebe uma consulta em texto livre via linha de comando e devolve o ranking híbrido top-k, sem depender de executar este notebook primeiro:

```bash
python demo.py "subgraph matching algorithms for graph databases" --k 10
```